# Fink/LSST — Zero-Point Proxy Heatmap on the LSSTCam Focal Plane

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`
and displays **two focal-plane heatmaps** in a single 1×2 figure:

| Panel | Quantity |
|---|---|
| Left  | Median of $\Delta m$ per CCD detector |
| Right | RMS (std) of $\Delta m$ per CCD detector |

### Zero-point proxy definition

$$\Delta m = 2.5 \log_{10}\!\left(\frac{F_{\rm psf}}{F_{\rm ap}}\right)$$

The quantity is computed **per individual src alert** (one row = one DIA source),
**independently of the diaObject** to which the source belongs.
Only rows where both `psfFlux > 0` and `apFlux > 0` are kept (non-detections excluded).

### Error propagation (for reference)
$$\sigma_{\Delta m} = \frac{2.5}{\ln 10}\,
\sqrt{\left(\frac{\sigma_{\rm psf}}{F_{\rm psf}}\right)^2
     + \left(\frac{\sigma_{\rm ap}}{F_{\rm ap}}\right)^2}$$

### Per-CCD aggregation
- **Median** $\langle\Delta m\rangle$ : robust central value → reveals systematic PSF-model offsets.
- **RMS / std** $\sigma(\Delta m)$ : scatter around the median → reveals temporal instability
  or spatial inhomogeneity within the CCD.

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Creation date:** 2026-04-02  
- **Last update:** 2026-04-02  
- **Subject:** Fink alert broker — Rubin LSST focal-plane ZP-proxy heatmap

## 1. Imports & configuration

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = "figs_FINK_BLOCK_LC_05b"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Rubin LSSTCam focal-plane geometry CSV ────────────────────────────────────
PATH_GEOM_CSV = "data_FocalPlane/ccd_geometry.csv"

# ── Groups to use for the ZP computation ─────────────────────────────────────
# Only photometrically stable Gaia stars — the same selection used in notebook 05.
GROUPS_OF_INTEREST = ["gaia_star_stable", "gaia_star_stable_hq"]

# ── Column names in the src parquet files ────────────────────────────────────
COL_DETECTOR = "r:detector"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_AP = "r:apFlux"
COL_APERR = "r:apFluxErr"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"

# ── ZP-proxy constant: 2.5 / ln(10) ─────────────────────────────────────────
MAGFAC = 2.5 / np.log(10.0)  # ≈ 1.0857

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": False,  # grid off for focal-plane maps
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Geometry CSV     : {os.path.abspath(PATH_GEOM_CSV)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"MAGFAC (2.5/ln10): {MAGFAC:.6f}")

## 2. Load src Parquet files and concatenate

In [ ]:
# Discover all src parquet files on disk
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path):
    return os.path.basename(path).replace("_src.parquet", "")


groups_src = {group_from_path(p): p for p in src_files}

print(f"src parquet files found on disk: {len(groups_src)}")
for g, p in groups_src.items():
    print(f"  {g}")

In [ ]:
# Load and concatenate only the groups of interest
dfs_to_concat = []

for group in GROUPS_OF_INTEREST:
    if group not in groups_src:
        print(f"  WARNING: group '{group}' not found on disk — skipped.")
        continue
    df_g = pd.read_parquet(groups_src[group])
    df_g["_group"] = group  # keep track of origin
    dfs_to_concat.append(df_g)
    print(f"  Loaded {group}: {len(df_g):,} rows")

if len(dfs_to_concat) == 0:
    raise RuntimeError(
        f"None of the requested groups {GROUPS_OF_INTEREST} were found in {DIR_DATA}. Run notebook 01 first."
    )

df_src = pd.concat(dfs_to_concat, ignore_index=True)
print(f"\nTotal src rows loaded : {len(df_src):,}")

## 3. Schema inspection & column validation

In [ ]:
print("Columns in src table:")
print(df_src.dtypes.to_string())

In [ ]:
# Validate required columns
required = [COL_DETECTOR, COL_PSF, COL_PSFERR, COL_AP, COL_APERR]
missing = [c for c in required if c not in df_src.columns]
if missing:
    raise KeyError(
        f"Missing required columns: {missing}\n"
        "Both psfFlux and apFlux (with errors) are needed for the ZP proxy."
    )
print("All required columns present.")
print(f"Unique detectors (CCDs) : {df_src[COL_DETECTOR].nunique()}")
print(f"Detector id range       : {df_src[COL_DETECTOR].min()} – {df_src[COL_DETECTOR].max()}")

## 4. Compute the ZP proxy per src row

$$\Delta m = 2.5 \log_{10}\!\left(\frac{F_{\rm psf}}{F_{\rm ap}}\right)$$

Rows where either flux is non-positive are masked to NaN and excluded from
the per-CCD statistics (non-detections, negative DIA residuals).

In [ ]:
def compute_zp_proxy_column(df):
    """
    Add a 'delta_m' column to df in-place.

    delta_m = 2.5 * log10(psfFlux / apFlux)

    Rows where psfFlux <= 0 or apFlux <= 0 receive NaN.

    Parameters
    ----------
    df : pd.DataFrame  — must contain COL_PSF and COL_AP

    Returns
    -------
    df : pd.DataFrame  — same dataframe with 'delta_m' column added
    """
    f_psf = df[COL_PSF].values.astype(float)
    f_ap = df[COL_AP].values.astype(float)

    # Valid mask: both fluxes must be finite and strictly positive
    valid = np.isfinite(f_psf) & (f_psf > 0) & np.isfinite(f_ap) & (f_ap > 0)

    delta_m = np.full(len(df), np.nan)
    delta_m[valid] = 2.5 * np.log10(f_psf[valid] / f_ap[valid])

    df = df.copy()
    df["delta_m"] = delta_m
    return df


df_src = compute_zp_proxy_column(df_src)

n_valid = df_src["delta_m"].notna().sum()
n_invalid = df_src["delta_m"].isna().sum()
print(f"delta_m computed : {n_valid:,} valid  |  {n_invalid:,} masked (non-positive flux)")
print(f"delta_m range    : [{df_src['delta_m'].min():.4f},  {df_src['delta_m'].max():.4f}] mag")
print(f"delta_m median   : {df_src['delta_m'].median():.4f} mag")

## 5. Per-CCD aggregation: median and RMS of Δm

In [ ]:
# Group by detector and compute robust statistics on delta_m
# Only use rows with valid (non-NaN) delta_m values.
df_valid = df_src.dropna(subset=["delta_m"])

ccd_stats = (
    df_valid.groupby(COL_DETECTOR)["delta_m"]
    .agg(
        n_src="count",
        dm_median="median",
        dm_mean="mean",
        dm_std="std",
    )
    .reset_index()
    .rename(columns={COL_DETECTOR: "detector"})
)

print(f"CCDs with at least one valid src : {len(ccd_stats)}")
print(ccd_stats.describe().to_string())

In [ ]:
print("Per-CCD ZP-proxy statistics (first 20 rows):")
display(ccd_stats.head(20))

## 6. Load focal-plane geometry

In [ ]:
if not os.path.exists(PATH_GEOM_CSV):
    raise FileNotFoundError(f"Focal-plane geometry CSV not found: {PATH_GEOM_CSV}")

geom = pd.read_csv(PATH_GEOM_CSV)
print(f"Geometry CSV loaded : {len(geom)} CCDs")
print(geom.columns.tolist())
print(geom.head(3).to_string())

In [ ]:
# Merge geometry with per-CCD statistics
geom_zp = geom.merge(ccd_stats[["detector", "n_src", "dm_median", "dm_std"]], on="detector", how="left")

# CCDs with no data stay NaN — they will be rendered in light grey
n_no_data = geom_zp["dm_median"].isna().sum()
print(f"CCDs with data    : {geom_zp['dm_median'].notna().sum()}")
print(f"CCDs without data : {n_no_data}  (will appear grey)")

## 7. Focal-plane heatmap function (1×2 dual panel)

The function draws both panels — median $\Delta m$ and RMS $\sigma(\Delta m)$ — side by side
on a shared figure, one CCD polygon per detector.

In [ ]:
def draw_focal_plane(
    ax,
    geom_df,
    value_col,
    cmap,
    norm,
    title,
    colorbar_label,
    fig,
    show_ccd_label=True,
):
    """
    Draw a single focal-plane heatmap panel on axes *ax*.

    Parameters
    ----------
    ax             : matplotlib Axes
    geom_df        : pd.DataFrame — geometry + merged statistics
    value_col      : str          — column to colour-code ('dm_median' or 'dm_std')
    cmap           : matplotlib colormap
    norm           : matplotlib Normalize / DivergingNorm instance
    title          : str          — subplot title
    colorbar_label : str          — label for the colour bar
    fig            : matplotlib Figure — needed to attach the colour bar
    show_ccd_label : bool         — whether to print detector id + name inside each patch
    """
    xmin = geom_df[[f"corner{i}_x" for i in range(4)]].min().min()
    xmax = geom_df[[f"corner{i}_x" for i in range(4)]].max().max()
    ymin = geom_df[[f"corner{i}_y" for i in range(4)]].min().min()
    ymax = geom_df[[f"corner{i}_y" for i in range(4)]].max().max()

    cmap_obj = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap

    for _, row in geom_df.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row[value_col]

        if np.isnan(val):
            # No data: render as light grey
            face_color = "lightgrey"
        else:
            face_color = cmap_obj(norm(val))

        poly = patches.Polygon(
            corners,
            facecolor=face_color,
            edgecolor="black",
            linewidth=0.2,
        )
        ax.add_patch(poly)

        if show_ccd_label:
            ax.text(
                row["x_center"],
                row["y_center"],
                f"{int(row['detector'])}\n{row['name']}",
                ha="center",
                va="center",
                fontsize=5.5,
                fontweight="bold",
                color="k",
            )

    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Focal plane X  [mm]", fontsize=9)
    ax.set_ylabel("Focal plane Y  [mm]", fontsize=9)
    ax.set_title(title, fontsize=10)

    # Colour bar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(colorbar_label, fontsize=8)


print("draw_focal_plane() defined.")

## 8. Plot: median Δm and RMS(Δm) on the focal plane (1×2)

- **Left panel** : median of $\Delta m$ per CCD — diverging colour scale centred on 0.
- **Right panel** : RMS $\sigma(\Delta m)$ per CCD — sequential colour scale (0 = perfect stability).
- Grey patches = CCDs with no valid src alert.

In [ ]:
# ── Colour-scale limits ──────────────────────────────────────────────────────
# Median panel: symmetric diverging scale centred on 0
med_vals = geom_zp["dm_median"].dropna()
vmax_med = float(np.percentile(np.abs(med_vals), 95)) if len(med_vals) > 0 else 0.05
vmax_med = max(vmax_med, 1e-4)  # guard against zero range

norm_med = mcolors.TwoSlopeNorm(vmin=-vmax_med, vcenter=0.0, vmax=vmax_med)
cmap_med = "RdBu_r"  # red = psf brighter than ap, blue = ap brighter

# RMS panel: sequential scale starting at 0
std_vals = geom_zp["dm_std"].dropna()
vmax_std = float(np.percentile(std_vals, 95)) if len(std_vals) > 0 else 0.05
vmax_std = max(vmax_std, 1e-4)

norm_std = mcolors.Normalize(vmin=0.0, vmax=vmax_std)
cmap_std = "plasma"  # low scatter = dark, high scatter = bright yellow

print(f"Median colour range : [{-vmax_med:.4f},  {vmax_med:.4f}] mag")
print(f"RMS    colour range : [0,  {vmax_std:.4f}] mag")

In [ ]:
# ── Draw the 1×2 dual focal-plane figure ─────────────────────────────────────
fig, (ax_med, ax_std) = plt.subplots(
    1,
    2,
    figsize=(18, 9),
    constrained_layout=True,
)

# Left panel: median Δm per CCD
draw_focal_plane(
    ax=ax_med,
    geom_df=geom_zp,
    value_col="dm_median",
    cmap=cmap_med,
    norm=norm_med,
    title=r"Median $\Delta m = 2.5\,\log_{10}(F_{\rm psf}/F_{\rm ap})$ per CCD  [mag]",
    colorbar_label=r"median $\Delta m$  [mag]",
    fig=fig,
    show_ccd_label=True,
)

# Right panel: RMS σ(Δm) per CCD
draw_focal_plane(
    ax=ax_std,
    geom_df=geom_zp,
    value_col="dm_std",
    cmap=cmap_std,
    norm=norm_std,
    title=r"RMS  $\sigma(\Delta m)$ per CCD  [mag]",
    colorbar_label=r"$\sigma(\Delta m)$  [mag]",
    fig=fig,
    show_ccd_label=True,
)

fig.suptitle(
    r"LSSTCam Focal Plane — ZP proxy $\Delta m = 2.5\,\log_{10}(F_{\rm psf}/F_{\rm ap})$ "
    f"(groups: {', '.join(GROUPS_OF_INTEREST)},  "
    f"N_src={n_valid:,} valid alerts)",
    fontsize=11,
)

savefig("focal_plane_ZP_median_rms")
plt.show()

## 9. Optional: per-band focal-plane heatmaps

The same 1×2 layout is produced for each photometric band `u g r i z y`.
This reveals whether PSF-model biases are chromatic (band-dependent).

In [ ]:
BANDS = list("ugrizy")

# Check that the band column is present
if COL_BAND not in df_src.columns:
    print(f"Column '{COL_BAND}' not found — skipping per-band focal-plane plots.")
else:
    for band in BANDS:
        df_band = df_src[df_src[COL_BAND] == band].dropna(subset=["delta_m"])

        if len(df_band) == 0:
            print(f"  band {band}: no valid src — skipped.")
            continue

        # Per-CCD statistics for this band
        band_stats = (
            df_band.groupby(COL_DETECTOR)["delta_m"]
            .agg(n_src="count", dm_median="median", dm_std="std")
            .reset_index()
            .rename(columns={COL_DETECTOR: "detector"})
        )

        geom_band = geom.merge(
            band_stats[["detector", "n_src", "dm_median", "dm_std"]],
            on="detector",
            how="left",
        )

        # Colour-scale limits for this band
        med_b = geom_band["dm_median"].dropna()
        std_b = geom_band["dm_std"].dropna()

        vmax_m = max(float(np.percentile(np.abs(med_b), 95)) if len(med_b) > 0 else 0.05, 1e-4)
        vmax_s = max(float(np.percentile(std_b, 95)) if len(std_b) > 0 else 0.05, 1e-4)

        norm_m = mcolors.TwoSlopeNorm(vmin=-vmax_m, vcenter=0.0, vmax=vmax_m)
        norm_s = mcolors.Normalize(vmin=0.0, vmax=vmax_s)

        fig_b, (ax_m, ax_s) = plt.subplots(
            1,
            2,
            figsize=(18, 9),
            constrained_layout=True,
        )

        draw_focal_plane(
            ax=ax_m,
            geom_df=geom_band,
            value_col="dm_median",
            cmap="RdBu_r",
            norm=norm_m,
            title=rf"Band {band} — Median $\Delta m$ per CCD  [mag]",
            colorbar_label=rf"median $\Delta m$  [mag]",
            fig=fig_b,
            show_ccd_label=True,
        )
        draw_focal_plane(
            ax=ax_s,
            geom_df=geom_band,
            value_col="dm_std",
            cmap="plasma",
            norm=norm_s,
            title=rf"Band {band} — RMS $\sigma(\Delta m)$ per CCD  [mag]",
            colorbar_label=rf"$\sigma(\Delta m)$  [mag]",
            fig=fig_b,
            show_ccd_label=True,
        )

        fig_b.suptitle(
            rf"Band {band} — LSSTCam Focal Plane ZP proxy "
            f"(N_src={len(df_band):,} valid alerts)",
            fontsize=11,
        )

        savefig(f"focal_plane_ZP_median_rms_band{band}")
        plt.show()

## 10. Summary table: per-CCD ZP statistics

In [ ]:
# Full summary: geometry + statistics merged
summary_cols = ["detector", "name", "x_center", "y_center", "n_src", "dm_median", "dm_mean", "dm_std"]

# Merge mean into geom_zp if not already there
geom_zp_full = geom.merge(
    ccd_stats[["detector", "n_src", "dm_median", "dm_mean", "dm_std"]],
    on="detector",
    how="left",
)

df_summary_ccd = geom_zp_full[summary_cols].sort_values("detector").reset_index(drop=True)
print("Per-CCD ZP proxy summary:")
display(df_summary_ccd)

In [ ]:
# Save summary to CSV
summary_path = os.path.join(DIR_FIGS, "ccd_zp_proxy_summary.csv")
df_summary_ccd.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")